# DSC 80 - Discussion 06
# Web Scraping: Meet the MOSAIC Lab

Scrape the [MOSAIC Lab webpage](https://mosaic-laboratory.github.io/) for **projects, publications, and team members**, then combine our steps into one reusable function.

## 1. Quick recap: HTML and the DOM

- **HTML:** content and tags.
- **CSS:** appearance, classes, and IDs.
- **JavaScript:** interactivity and dynamically loaded content.
- **BeautifulSoup:** parses HTML into a searchable tree.

**Load → locate data → extract fields → DataFrame**

### HTML as a DOM tree
<img src = 'imgs/dom.jpg'></img>

### Common BeautifulSoup methods

| Method | Input | Returns | Example |
| --- | --- | --- | --- |
| `find()` | Tag name and attributes | First match, or `None` | `soup.find('h2', class_='title')` |
| `find_all()` | Tag name and attributes | All matches (possibly empty) | `soup.find_all('h2', class_='title')` |
| `select_one()` | CSS selector | First match, or `None` | `soup.select_one('h2.title')` |
| `select()` | CSS selector | All matches (possibly empty) | `soup.select('h2.title')` |

**CSS syntax belongs in `select` / `select_one`.** 

Read text with `tag.get_text(' ', strip=True)`: `' '` separates text fragments, and `strip=True` trims their edges. Read an attribute with `tag.get('href')`.

## 2. Load the saved HTML

**Useful tip:** `requests` gets the initial HTML, but this site's cards are added by JavaScript. Copy the completed HTML from the browser:

1. Open the web page and wait for the cards to appear.
2. In **Inspect → Elements**, right-click the top-level `<html>` element → **Copy → Copy outerHTML**.
3. Save as `mosaiclab_inspect_html.txt` beside this notebook.

> Copy from **Elements**, not **View page source**. BeautifulSoup parses this saved HTML snapshot.

In [1]:
import pandas as pd
from bs4 import BeautifulSoup

HTML_FILE = 'mosaiclab_inspect_html.txt'

In [2]:
def load_soup(filename):
    """Read saved HTML into a BeautifulSoup tree.

    Args:
        filename (str): Path to the text file containing HTML.

    Returns:
        BeautifulSoup: Parsed page.
    """
    with open(filename, encoding='utf-8') as file:
        html = file.read()
    return BeautifulSoup(html, 'html.parser')


soup = load_soup(HTML_FILE)
soup.title.get_text(' ', strip=True)

'MOSAIC Laboratory'

## 3. Scrape projects

The HTML examples below omit layout-only wrappers.

```html
<section id="projects">
  <article>
    <p class="post-category">Wearable Devices | Health</p>
    <h2 class="title">
      <a href="projects-details.html?...">Project title</a>
    </h2>
    <div class="post-meta">
      <p class="post-author">Project description</p>
      <p class="post-date">On going</p>
    </div>
  </article>
  <!-- More project cards... -->
</section>
```

Find the **section by ID**, then its **cards by tag**. Inspect the first card.

In [3]:
project_section = soup.find('section', id='projects')
project_cards = project_section.find_all('article')
print('Number of projects:', len(project_cards))
project_cards[0]

Number of projects: 5


<article><div class="post-img"><a href="projects-details.html?project_name=mobile_health"><img alt="" class="img-fluid" loading="lazy" src="assets/img/projects/mobile_health.png"/></a></div><p class="post-category">Wearable Devices | Smart Health Monitoring</p><h2 class="title"><a href="projects-details.html?project_name=mobile_health">Mobile Health Sensing</a></h2><div class="d-flex align-items-center"><div class="post-meta"><p class="post-author">Developing wearable and mobile systems to monitor health outcomes. Our work focuses on creating wearable devices that provide accurate, real-time health monitoring to support medical decisions.</p><p class="post-date">On going</p></div></div></article>

In [4]:
# Extract the title and keywords from one card.
first_project = project_cards[0]
title = first_project.find('h2', class_='title')
print(title.get_text(' ', strip=True))
print(first_project.find('p', class_='post-category').get_text(' ', strip=True))

Mobile Health Sensing
Wearable Devices | Smart Health Monitoring


### One card → one dictionary

As in Lecture 10's `process_quote`, write a helper for one record.

In [5]:
def process_project(article):
    """Extract one project's fields.

    Args:
        article (Tag): One project card.

    Returns:
        dict: Project title, description, keywords, and status.
    """
    title = article.find('h2', class_='title')
    return {
        'Project': title.get_text(' ', strip=True),
        'Description': article.find('p', class_='post-author').get_text(' ', strip=True),
        'Keywords': article.find('p', class_='post-category').get_text(' ', strip=True),
        'Status': article.find('p', class_='post-date').get_text(' ', strip=True),
    }


process_project(first_project)

{'Project': 'Mobile Health Sensing',
 'Description': 'Developing wearable and mobile systems to monitor health outcomes. Our work focuses on creating wearable devices that provide accurate, real-time health monitoring to support medical decisions.',
 'Keywords': 'Wearable Devices | Smart Health Monitoring',
 'Status': 'On going'}

### All cards → a DataFrame

Each dictionary becomes a row; its keys become column names.

In [6]:
project_rows = [process_project(card) for card in project_cards]
projects = pd.DataFrame(project_rows)
projects.head()

,Project,Description,Keywords,Status
0,Mobile Health Sensing,Developing wearable and mobile systems to moni...,Wearable Devices | Smart Health Monitoring,On going
1,Cognitive and Behavioral Health,Exploring the interaction between cognitive pr...,Machine Learning | Applied AI,On going
2,Digital Epidemiology,Researching health trends and disease outbreak...,Machine Learning | Contactless Sensing | Healt...,On going
3,Signal Processing and AI,Leveraging AI and signal processing for ubiqui...,Signal Processing | Deep Learning,On going
4,Interaction Systems,Designing human-centered systems that integrat...,Sensors | HCI,On going


## 4. Scrape publications


```html
<section id="publications">
  <div id="publication_list">
    <div>
      <h2 class="publication-year-title">2026</h2>
      <div class="publication-row">
        <div class="publication-content">
          <p class="publication-title"><b>Paper title</b></p>
          <p>Author names</p>
          <p class="publication-venue">Conference or journal</p>
          <div class="publication-links">
            <a href="https://example.com/paper">Paper</a>
          </div>
        </div>
      </div>
      <!-- More publication cards... -->
    </div>
  </div>
</section>
```

The authors are in the second paragraph of `publication-content`. The first `<a>` inside `publication-links` gives the resource link (usually **Paper**; one is **Poster**).

For the year, `find_previous()` finds the nearest earlier matching heading. 

In [7]:
publication_section = soup.find('section', id='publications')
publication_cards = publication_section.find_all('div', class_='publication-row')
print('Number of publications:', len(publication_cards))
publication_cards[0].find('div', class_='publication-content')

Number of publications: 46


<div class="publication-content"><p class="publication-title"><b>From Wearable Data to Risk Literacy: Visualizing Physiological Heat Stress for Rickshaw Pullers in Dhaka</b></p><p>Masfiqur Rahaman, Shahad Shahriar Rahman, MD Sajid Mostafiz Noor, Khandaker Shadman Sakib Siddiquee, Mohammad Kamrul Hasan, Conan Minihan, Manas Satish Bedmutha, Shamil Zainuddin, Benjamin Smarr, A. B. M. Alim Al Islam, and Tauhidur Rahman</p><p class="publication-venue">In Proceedings of the 2026 ACM International Symposium on Wearable Computers (ISWC 2026)</p><div class="publication-links"><a href="/assets/papers/RP_Interview_Paper___ISWC (15).pdf" rel="noopener noreferrer" target="_blank">Paper</a></div></div>

In [8]:
def process_publication(div):
    """Extract one publication's fields.

    Args:
        div (Tag): One publication card.

    Returns:
        dict: Title, authors, venue, year heading, and first resource link.
    """
    content = div.find('div', class_='publication-content')
    paragraphs = content.find_all('p')
    year = div.find_previous('h2', class_='publication-year-title')
    link = div.find('div', class_='publication-links').find('a')
    return {
        'Title': content.find('p', class_='publication-title').get_text(' ', strip=True),
        'Authors': paragraphs[1].get_text(' ', strip=True),
        'Venue': content.find('p', class_='publication-venue').get_text(' ', strip=True),
        'Year': year.get_text(' ', strip=True),
        'Link': link.get('href') if link is not None else None,
    }


publications = pd.DataFrame([process_publication(card) for card in publication_cards])
publications.head()

,Title,Authors,Venue,Year,Link
0,From Wearable Data to Risk Literacy: Visualizi...,"Masfiqur Rahaman, Shahad Shahriar Rahman, MD S...",In Proceedings of the 2026 ACM International S...,2026,/assets/papers/RP_Interview_Paper___ISWC (15).pdf
1,Toward Foundation Model for Multivariate Weara...,"Yunfei Luo, Yuliang Chen, Asif Salekin, and Ta...",ACM Transactions on Computing for Healthcare (...,2026,https://dl.acm.org/doi/10.1145/3803808
2,Personalized entropy-informed deep learning fo...,"Yunfei Luo, Iman Deznabi, Bhanu Gullapalli, Ma...",Nature Mental Health,2026,https://www.nature.com/articles/s44220-025-005...
3,OmniFlow: A Framework for Generalizable Surrog...,"M Tanjid Hasan Tonmoy, Upal Mahbub, and Tauhid...",BuildSys 2025,2025,https://dl.acm.org/doi/10.1145/3736425.3770117
4,Tandem: Behavior Assessment Using Multimodal S...,"Manasa Kalanadhabhatta, Tauhidur Rahman, Adam ...","Proceedings of the ACM on Interactive, Mobile,...",2025,https://dl.acm.org/doi/10.1145/3770705


## 5. Scrape team members

```html
<section id="teams">
  <!-- Other layout containers omitted -->
  <div class="team-category">
    <h3 class="category-title">Current Students</h3>
    <div class="row">
      <div class="member">
        <div class="member-info">
          <h4>Member name</h4>
          <span>PhD Student</span>
          <p>Email: <a href="mailto:name@ucsd.edu">name@ucsd.edu</a></p>
        </div>
      </div>
      <!-- More members in this category... -->
    </div>
  </div>
</section>
```

Find `#teams`, then each `div` with class `member`. Some members have no email link, so check for `None`.

`find_previous()` finds the nearest earlier `h3` with class `category-title`. Its text gives the member category.

In [9]:
team_section = soup.find('section', id='teams')
member_cards = team_section.find_all('div', class_='member')
print('Number of members:', len(member_cards))
member_cards[0].find('div', class_='member-info')

Number of members: 21


<div class="member-info text-center"><h4>Tauhidur Rahman</h4><span>Assistant Professor<br/></span><p>Halıcıoğlu Data Science Institute, UCSD</p><p>Email: <a href="mailto:trahman@ucsd.edu">trahman@ucsd.edu</a></p></div>

In [10]:
def process_member(div):
    """Extract one team member's fields.

    Args:
        div (Tag): One member card.

    Returns:
        dict: Name, role, category, and email (None when absent).
    """
    info = div.find('div', class_='member-info')
    email = info.find('a')
    category = div.find_previous('h3', class_='category-title')
    return {
        'Name': info.find('h4').get_text(' ', strip=True),
        'Role': info.find('span').get_text(' ', strip=True),
        'Category': category.get_text(' ', strip=True),
        'Email': email.get_text(' ', strip=True) if email is not None else None,
    }


team = pd.DataFrame([process_member(card) for card in member_cards])
team.head()

,Name,Role,Category,Email
0,Tauhidur Rahman,Assistant Professor,Principal Investigator,trahman@ucsd.edu
1,Caden Stewart,PhD Student,Current Students,None
2,Conan Minihan,PhD Student,Current Students,None
3,Tamar Schaap,PhD Student,Current Students,None
4,Masfiqur Rahaman,PhD Student,Current Students,None


## 6. Generalize: find cards, then fields

One config describes **which cards to collect** and **what to read inside each card**.

### CSS selector quick guide

| Syntax | Meaning | Example |
| --- | --- | --- |
| `tag` | Tag name | `article` |
| `#id` | An element with this ID | `#projects` |
| `.class` | Elements with this class | `.publication-row` |
| `tag.class` | Both the tag and class match | `h2.title` |
| `A B` | B inside A, at any depth | `.publication-links a` |
| `A + B` | B immediately follows A under the same parent | `.publication-title + p` |

In [11]:
# Config guide:
# filename: saved HTML file.
# cards: CSS selector for the repeated records.
# fields: output column names and their extraction settings:
#   selector: CSS selector inside each card.
#   previous: instead, find an earlier heading matching these HTML attributes.
#   attribute: optional attribute to read, e.g. 'href'; otherwise read text.
# Missing fields become None. Links stay exactly as written in the HTML.

project_config = {
    'filename': HTML_FILE,
    'cards': '#projects article',
    'fields': {
        'Project': {
            'selector': 'h2.title',
        },
        'Description': {
            'selector': 'p.post-author',
        },
        'Keywords': {
            'selector': 'p.post-category',
        },
        'Status': {
            'selector': 'p.post-date',
        },
    },
}

### Read one field

`select_one()` finds the first matching element inside a card. For shared headings, `find_previous()` searches backward in document order.

In [12]:
def read_field(card, field):
    """Read one configured value from a card.

    Args:
        card (Tag): One record's HTML node.
        field (dict): Selector or previous-heading attributes; optional attribute.

    Returns:
        str or None: Text, an attribute value, or a missing value.
    """
    # fetch the element
    if 'previous' in field:
        value = card.find_previous(attrs=field['previous'])
    else:
        value = card.select_one(field['selector'])

    # fetch text content or the attribute content
    if value is not None and 'attribute' in field:
        value = value.get(field['attribute'])

    # formatting
    if value is None or isinstance(value, str):
        return value
    return value.get_text(' ', strip=True)

### Collect cards → build a DataFrame

Each card produces one dictionary (one row). The field names become the columns.

In [13]:
def scrape_table(config):
    """Read the configured cards and fields from saved HTML.

    Args:
        config (dict): Filename, card selector, and field settings.

    Returns:
        pd.DataFrame: One row per card.
    """
    # load the raw html into soup object
    soup = load_soup(config['filename'])

    # fetch the list element
    cards = soup.select(config['cards'])

    # iterate over each element
    rows = []
    for card in cards:
        row = {}

        # iterate over desired fields
        for column, field in config['fields'].items():
            row[column] = read_field(card, field)
        rows.append(row)

    # construct dataframe object and return
    return pd.DataFrame(rows, columns=config['fields'])


projects = scrape_table(project_config)
projects.head()

,Project,Description,Keywords,Status
0,Mobile Health Sensing,Developing wearable and mobile systems to moni...,Wearable Devices | Smart Health Monitoring,On going
1,Cognitive and Behavioral Health,Exploring the interaction between cognitive pr...,Machine Learning | Applied AI,On going
2,Digital Epidemiology,Researching health trends and disease outbreak...,Machine Learning | Contactless Sensing | Healt...,On going
3,Signal Processing and AI,Leveraging AI and signal processing for ubiqui...,Signal Processing | Deep Learning,On going
4,Interaction Systems,Designing human-centered systems that integrat...,Sensors | HCI,On going


### Publications

`.publication-title + p` selects the paragraph immediately after the title. The year comes from the preceding year heading.

In [14]:
publication_config = {
    'filename': HTML_FILE,
    'cards': '#publications div.publication-row',
    'fields': {
        'Title': {
            'selector': 'p.publication-title',
        },
        'Authors': {
            'selector': '.publication-title + p',
        },
        'Venue': {
            'selector': 'p.publication-venue',
        },
        'Year': {
            'previous': {
                'class': 'publication-year-title',
            },
        },
        'Link': {
            'selector': '.publication-links a',
            'attribute': 'href',
        },
    },
}

publications = scrape_table(publication_config)
publications.head()

,Title,Authors,Venue,Year,Link
0,From Wearable Data to Risk Literacy: Visualizi...,"Masfiqur Rahaman, Shahad Shahriar Rahman, MD S...",In Proceedings of the 2026 ACM International S...,2026,/assets/papers/RP_Interview_Paper___ISWC (15).pdf
1,Toward Foundation Model for Multivariate Weara...,"Yunfei Luo, Yuliang Chen, Asif Salekin, and Ta...",ACM Transactions on Computing for Healthcare (...,2026,https://dl.acm.org/doi/10.1145/3803808
2,Personalized entropy-informed deep learning fo...,"Yunfei Luo, Iman Deznabi, Bhanu Gullapalli, Ma...",Nature Mental Health,2026,https://www.nature.com/articles/s44220-025-005...
3,OmniFlow: A Framework for Generalizable Surrog...,"M Tanjid Hasan Tonmoy, Upal Mahbub, and Tauhid...",BuildSys 2025,2025,https://dl.acm.org/doi/10.1145/3736425.3770117
4,Tandem: Behavior Assessment Using Multimodal S...,"Manasa Kalanadhabhatta, Tauhidur Rahman, Adam ...","Proceedings of the ACM on Interactive, Mobile,...",2025,https://dl.acm.org/doi/10.1145/3770705


### Members

Use the same preceding-heading lookup for the member category.

In [15]:
team_config = {
    'filename': HTML_FILE,
    'cards': '#teams div.member',
    'fields': {
        'Name': {
            'selector': '.member-info h4',
        },
        'Role': {
            'selector': '.member-info span',
        },
        'Category': {
            'previous': {
                'class': 'category-title',
            },
        },
        'Email': {
            'selector': '.member-info a',
        },
    },
}

team = scrape_table(team_config)
team.head()

,Name,Role,Category,Email
0,Tauhidur Rahman,Assistant Professor,Principal Investigator,trahman@ucsd.edu
1,Caden Stewart,PhD Student,Current Students,None
2,Conan Minihan,PhD Student,Current Students,None
3,Tamar Schaap,PhD Student,Current Students,None
4,Masfiqur Rahaman,PhD Student,Current Students,None
